In [30]:
FILE_PATH = 'C://Users//User//Downloads//bitmex_data_1m.csv'
#FILE_PATH = 'D://bitmex_data_1m.csv'

from abc import ABC, abstractmethod
from common import *
import plotly
import pandas as pd

In [31]:
test_df = pd.read_csv(FILE_PATH, delimiter=',')
test_df

,timestamp,symbol,trades,volume,turnover,homeNotional,foreignNotional,open,high,low,close,vwap,lastSize
0,2015-09-25 12:01:00+00:00,XBTUSD,0,0,0,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-09-25 12:02:00+00:00,XBTUSD,0,0,0,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-09-25 12:03:00+00:00,XBTUSD,0,0,0,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-09-25 12:04:00+00:00,XBTUSD,0,0,0,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-09-25 12:05:00+00:00,XBTUSD,0,0,0,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4967883,2025-03-06 10:04:00+00:00,XBTUSD,18,21700,23933086,0.239331,21700.0,90667.9,90716.2,90629.4,90707.0,90670.0517,100.0
4967884,2025-03-06 10:05:00+00:00,XBTUSD,21,78800,86837110,0.868371,78800.0,90707.0,90766.7,90707.0,90758.1,90745.7486,900.0
4967885,2025-03-06 10:06:00+00:00,XBTUSD,29,59100,65115360,0.651154,59100.0,90758.1,90798.1,90740.2,90760.8,90763.0449,9200.0
4967886,2025-03-06 10:07:00+00:00,XBTUSD,36,123200,135651914,1.356519,123200.0,90760.8,90840.0,90800.0,90818.8,90821.5719,300.0


In [32]:
import numpy as np

In [33]:


# 데이터 로드 전략 인터페이스
class DataLoaderStrategy(ABC):
    @abstractmethod
    def load_data(self):
        #1분봉 데이터를 불러오는 과정
        pass
    
    #@abstractmethod
    #def pre_precessing(self, df, base_delta, from_date=None):
    #    pass
    #    #1분봉 데이터를 여러 분봉으로 변경하는 함수
        

# 구체적인 데이터 로드 전략: CSV 로드
class BitmexCSVDataLoader(DataLoaderStrategy):
    
    raw_data = None #1분봉 df를 담을 변수
    final_df = None
    
    def __init__(self, base_delta: int, from_date:str = None):
        self.base_delta = base_delta  # 인스턴스 변수로 저장
        self.from_date = from_date
    
    def load_data(self):
        print("CSV 데이터를 로드하고 Nan을 제거합니다.")
        import pandas as pd
        df = pd.read_csv(FILE_PATH, delimiter=',')
        df = df.dropna()
        
        print(f"입력받은 {self.base_delta}분봉으로 {self.from_date} 부터 표현합니다.")
        
        if(self.from_date):
            df = df.loc[df.timestamp >= self.from_date]  #일단 GMT 니까 1분 뒤로 조정할 걸 생각하고 +1분부터 가져오면 된다.
        
        print("CSV 데이터 로드 완료.")
        
        df=df[['timestamp','high','low','open','close']]
        
        #timestamp 를 kst 로 조정
        df['timestamp_kst'] = df['timestamp'].apply(convert_gmt_to_kst)
        
        #클래스에 세팅
        raw_data = df
        
        df = self.__pre_precessing(df, self.base_delta, self.from_date)
        print(f"전처리 완료")

        return df
    
    def __pre_precessing(self, df, base_delta , from_date=None):
        df = df[['timestamp_kst','open','low','high','close']]
    
        df = df.reset_index()
        
        #int 형태의 timestamp 열도 추가
        df['timestamp_int'] = df['timestamp_kst'].apply(convert_to_timestamp)
        
        #다시 필요한 컬럼만 정돈
        df = df[['timestamp_kst','timestamp_int','open', 'low','high','close']]
        
        #최종 x분봉의 형태구현 base_delta = 15, 45, 240, 1440
        final_df = df[['low']].rolling(window=base_delta).min()     
        final_df['open'] = df[['open']].rolling(window=base_delta).apply(lambda x: x[0], raw=True)
        final_df['high'] = df[['high']].rolling(window=base_delta).max() 
        final_df['close'] = df['close']                         
        final_df['timestamp_int'] = df[['timestamp_int']]-(base_delta*60-60)  
        
        final_df = final_df[base_delta-1:] #window수 -1 값만큼 버리고
        final_df['timestamp_kst'] = final_df['timestamp_int'].apply(convert_timestamp_to_datetime_str)
        final_df = final_df.reset_index()[['timestamp_kst','open','low','high','close','timestamp_int']] #리셋재구성
        
        final_df = final_df[final_df['timestamp_int']%(base_delta*60)==0]
        
        return final_df
        

# 데이터 처리 전략 인터페이스
class DataProcessingStrategy(ABC):
    @abstractmethod
    def process_data(self, data):
        pass

# 구체적인 데이터 처리 전략: 이동 평균 계산
class MovingAverageProcessing(DataProcessingStrategy):
    def process_data(self, data):
        print("이동 평균을 계산했습니다.")
        data['ma_20']=self.__sma(data,20)
        data['ma_60']=self.__sma(data,60)
        
        return data#[sum(data[:i+1])/(i+1) for i in range(len(data))]
    
    def __sma(self, data, period=20):
        return data['close'].rolling(window=period, min_periods=1).mean()
    
# 구체적인 데이터 처리 전략: RSI 계산
class RSIProcessing(DataProcessingStrategy):
    def process_data(self, data):
        data['RSI'] = self.__rsi(data)
        print("RSI를 계산했습니다.")
        
        return data#[sum(data[:i+1])/(i+1) for i in range(len(data))]
    
    def __rsi(self, data, period=14):
    
        import numpy as np
        
        delta = data['close'].diff(1)  # 종가의 변화량 계산
        gain = np.where(delta > 0, delta, 0)  # 상승분
        loss = np.where(delta < 0, -delta, 0)  # 하락분
    
        avg_gain = pd.Series(gain).rolling(window=period, min_periods=1).mean()
        avg_loss = pd.Series(loss).rolling(window=period, min_periods=1).mean()
        
        rs = avg_gain / (avg_loss + 1e-10)  # 0으로 나누는 오류 방지
        rsi = 100 - (100 / (1 + rs))
        
        rsi.index = data.index
        
        return rsi

#고점을 찾는 처리 전략
class HighPointScoringProcessing(DataProcessingStrategy):
    '''
    메인 df에 'high_score' 라는 컬럼을 추가하고, 외부 변수의 리스트로 들어온 
    밴드 값을 shift 하면서 그중에 가장 큰 가격에 +1 스코어를 한다.
    '''
    
    
    def __init__(self, bandwidth_list):
        self.bandwith_list = bandwidth_list
    
    def process_data(self, data):
        data = self.__get_high_score_by_list(data,self.bandwith_list)
        return data
    
    def __get_high_score_by_list(self, origin_df,lst):
        if(len(lst)==0):
            print("대상이 없습니다.")
            
        else:
            return_df = None
            for idx, i in enumerate(lst):
                if(idx==0):
                    return_df = self.__get_high_score_with_bandwidth(origin_df, 'high', i)
                else:
                    return_df = self.__get_high_score_with_bandwidth(return_df, 'high', i, reset=False)
    
        return return_df
    
    
        
    def __get_high_score_with_bandwidth(self, target_df, column_name, bandwidth, reset=True):
        '''
        df를 제공하면서 밴드 값을 같이 제공하면 이를 반복문으로 돌아가면서 score 를 쌓는 함수
        '''
        #일단 들어온 df 에 high_score 라는 컬럼 값 기본 강제 삽입
        if(reset==True):
            target_df['high_score']=0
        end_index = len(target_df) - bandwidth
        for idx,i in enumerate(range(end_index+1)):
            
            max_index = target_df.iloc[i:i+bandwidth][column_name].idxmax()
            
            #print(f"진입한 시작 값 : {i}~{i+bandwidth}")
            #print(f"가장 큰 인덱스 : {max_index}")
            #print(f"이때의 이미 기록된 값:{target_df.loc[max_index, 'high_score']} ")
            #print(f"이때의 이미 기록될 값:{target_df.loc[max_index]['high_score']+1} ")
            #print(max_index)
        #
            #print(f"세팅전 값 : {target_df.loc[max_index]['high_score']}")
        #
            #print(f"체크값 :{target_df.loc[max_index]['high_score']+1}")
            #
            target_df.loc[max_index,'high_score'] = target_df.loc[max_index]['high_score']+1
            #print(f"세팅후 값 : {target_df.loc[max_index]['high_score']}")
        
        print(f"수행한 숫자:{end_index}")
        print(f"가장 높은 점수:{target_df.loc[target_df['high_score'].idxmax()]['high_score']}")
        
        return target_df

#저점을 찾는 처리 전략
class LowPointScoringProcessing(DataProcessingStrategy):
    def __init__(self, bandwidth_list):
        self.bandwith_list = bandwidth_list
        
    def process_data(self, data):
        data = self.__get_low_score_by_list(data, self.bandwith_list)
        return data
    
    def __get_low_score_by_list(self,origin_df,lst):
        if(len(lst)==0):
            print("대상이 없습니다.")
            
        else:
            return_df = None
            for idx, i in enumerate(lst):
                if(idx==0):
                    return_df = self.__get_low_score_with_bandwidth(origin_df, 'low', i)
                else:
                    return_df = self.__get_low_score_with_bandwidth(return_df, 'low', i, reset=False)
        
        return return_df
    
    
    def __get_low_score_with_bandwidth(self,target_df, column_name, bandwidth, reset=True):
    
    
        #일단 들어온 df 에 high_score 라는 컬럼 값 기본 강제 삽입
        if(reset==True):
            target_df['low_score']=0
        end_index = len(target_df) - bandwidth
        for idx,i in enumerate(range(end_index+1)):
            
            min_index = target_df.iloc[i:i+bandwidth][column_name].idxmin()
            
            #print(f"진입한 시작 값 : {i}~{i+bandwidth}")
            #print(f"가장 큰 인덱스 : {max_index}")
            #print(f"이때의 이미 기록된 값:{target_df.loc[max_index, 'high_score']} ")
            #print(f"이때의 이미 기록될 값:{target_df.loc[max_index]['high_score']+1} ")
            #print(max_index)
            
            #print(f"세팅전 값 : {target_df.loc[max_index]['high_score']}")
            
            #print(f"체크값 :{target_df.loc[max_row_index]['high_score']+1}")
            
            target_df.loc[min_index,'low_score'] = target_df.loc[min_index]['low_score']+1
            #print(f"세팅후 값 : {target_df.loc[max_index]['high_score']}")
            

        return target_df

# 고점만을 필터팅하는 처리 전략
class GetHighPoints(DataProcessingStrategy):
    def __init__(self, data, target_column, threshold_ratio):
        self.data = data
        self.target_column = target_column
        self.threshold_ratio = threshold_ratio
        
    def process_data(self, data):
        #일단 0점인건 전부 제외
        not_high_score_zero_data =data.loc[data['high_score']!=0]
        final_high_point=self.__apply_threshold_with_normalizing_for_high_value(self.data,self.target_column,self.threshold_ratio)
        
        return final_high_point
        
    #df와 타겟 컬럼명을 받아서 해당 %이상의 값만 필터링하는 함수
    def __apply_threshold_with_normalizing_for_high_value(self, target_df, target_column, threshold):
    
        #min-max 정규화
        target_df['normalized_value_high'] = (target_df[target_column] - target_df[target_column].min()) / (target_df[target_column].max() - target_df[target_column].min())
        
        threshold = target_df['normalized_value_high'].quantile(threshold)
        print(f"정형화된 기준값은 : {threshold}")
        
        #상위 5%에 포함되는 애들을 별도로 선언
    
        df_calculated = target_df[target_df['normalized_value_high'] >= threshold]
        df_calculated.loc[df_calculated['normalized_value_high'] >= threshold, 'origin_high_score'] = df_calculated[target_column]
        #그래프를 그리기 위한 수단을 벌써 넣으면 안됨
        df_calculated.loc[df_calculated['normalized_value_high'] >= threshold, 'high_for_graph'] = df_calculated['high']
        
        import numpy as np
        df_calculated.loc[df_calculated['normalized_value_high'] < threshold, target_column] = np.nan
        
        return df_calculated
    
    def __merge_high_score_df(self, df_origin, base_score):
        df_calculated=df_origin.loc[df_origin['high_score'] > base_score]
        df_calculated.loc[df_calculated['high_score'] > base_score, 'origin_score'] = df_calculated['high_score']
        df_calculated.loc[df_calculated['high_score'] > base_score, 'high_score'] = df_calculated['high']
    
        import numpy as np
        df_calculated.loc[df_calculated['high_score'] <= base_score, 'high_score'] = np.nan
        
        return df_calculated
    
# 저점만을 필터팅하는 처리 전략
class GetLowPoints(DataProcessingStrategy):
    def __init__(self, data, target_column, threshold_ratio):
        self.data = data
        self.target_column = target_column
        self.threshold_ratio = threshold_ratio
        
    
    def process_data(self, data):
        #일단 0점인건 전부 제외
        not_high_score_zero_data =data.loc[data['low_score']!=0]
        final_low_point=self.__apply_threshold_with_normalizing_for_low_value(self.data,self.target_column,self.threshold_ratio)
        
        return final_low_point
    
    def __apply_threshold_with_normalizing_for_low_value(self, target_df, target_column, threshold):
        #min-max 정규화
        target_df['normalized_value_low'] = (target_df[target_column] - target_df[target_column].min()) / (target_df[target_column].max() - target_df[target_column].min())
        
        threshold = target_df['normalized_value_low'].quantile(threshold)
        print(f"정형화된 기준값은 : {threshold}")
        
        #threshold 로 적은 수 이하로 포함되는 애들을 별도로 선언
    
        df_calculated = target_df[target_df['normalized_value_low'] >= threshold]
        df_calculated.loc[df_calculated['normalized_value_low'] >= threshold, 'origin_low_score'] = df_calculated[target_column]
        #그래프를 그리기 위한 수단을 벌써 넣으면 안됨
        df_calculated.loc[df_calculated['normalized_value_low'] >= threshold, 'low_for_graph'] = df_calculated['low']
        
        import numpy as np
        df_calculated.loc[df_calculated['normalized_value_low'] < threshold, target_column] = np.nan
        
        return df_calculated
    
    def __merge_low_score_df(self, df_origin, base_score):
        df_calculated=df_origin.loc[df_origin['low_score'] > base_score]
        df_calculated.loc[df_calculated['low_score'] > base_score, 'origin_score'] = df_calculated['low_score']
        df_calculated.loc[df_calculated['low_score'] > base_score, 'low_score'] = df_calculated['low']
    
        import numpy as np
        df_calculated.loc[df_calculated['low_score'] <= base_score, 'low_score'] = np.nan
        
        return df_calculated
    
    
# 데이터 시각화 인터페이스
class Visualization(ABC):
    @abstractmethod
    def visualize(self):
        pass

    def add_trace(self, trace):
        self.price_trace_list.append(trace)
    

# 구체적인 시각화 전략: 라인 그래프
import plotly.graph_objects as go
from plotly.subplots import make_subplots

class BasicPriceWithRsiVisualization(Visualization):
    
    def __init__(self):
        self.price_trace_list = []
        self.rsi_trace_lsit = []
        
    def set_data(self, df, high_points_df, low_points_df):
        self.__df = df
        self.__high_points_df = high_points_df
        self.__low_points_df = low_points_df
        
    def visualize(self):
        self.__get_base_figure()
        self.__main_figure.show()
        
        
        
    def __get_base_figure(self):
        main_trace = go.Candlestick(
            x=list(self.__df['timestamp_kst']),
            open=list(self.__df['open']),
            high=list(self.__df['high']),
            low=list(self.__df['low']),
            close=list(self.__df['close']),
        )
        
        #두번째 고가 점 트레이스 만들기
        high_point_trace = go.Scatter(y=list(self.__high_points_df['high_for_graph']), x=list(self.__high_points_df['timestamp_kst']), 
        marker=dict(
        color='rgba(255, 0, 255, 1)',  # 점의 색상 설정 (RGBa 형식)
        size=5,  # 점의 크기 설정
        ),
        mode='markers', name='고점 그래프')
        
        #세번째 저가 점 트레이스 만들기
        low_point_trace = go.Scatter(y=list(final_low_point['low_for_graph']), x=list(final_low_point['timestamp_kst']), 
        marker=dict(
        color='rgba(0, 0, 0, 1)',  # 점의 색상 설정 (RGBa 형식)
        size=5,  # 점의 크기 설정
        ),
        mode='markers', name='저점 그래프')
        
        ma_20_trace = go.Scatter(x=self.__df['timestamp_kst'], y=self.__df['ma_20'], mode='lines', name='MA_20', line=dict(color='blue'))

        ma_60_trace = go.Scatter(x=self.__df['timestamp_kst'], y=self.__df['ma_60'], mode='lines', name='MA_60', line=dict(color='black'))
        
        rsi_trace = go.Scatter(x=self.__df['timestamp_kst'], y=self.__df['RSI'], mode='lines', name='RSI', line=dict(color='blue'))
        
        main_figure = self.__draw_subplots([main_trace,high_point_trace,low_point_trace,ma_20_trace,ma_60_trace],[rsi_trace])
        
        self.__main_figure = main_figure
   
    
    def __draw_subplots(self, price_trace_list, rsi_trace_list):
        
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                        row_heights=[0.7, 0.3],  # 위쪽(가격) 70%, 아래쪽(RSI) 30%
                        subplot_titles=("가격 차트", "RSI (14)"))
        
        fig.update_layout(
        title='Candlestick Chart',
        xaxis_title='Date',
        yaxis_title='Price',
        xaxis_rangeslider_visible=False,
        width=1000,  # 그래프 너비 설정
        height=800   # 그래프 높이 설정
        )
        
        #trace_list1를 전부 담음
        for i in price_trace_list:
            fig.add_trace(i, row=1, col=1)
        
        for i in rsi_trace_list:
            fig.add_trace(i, row=2, col=1)
            fig.add_hline(y=70, line_dash="dash", line_color="red", annotation_text="Overbought (70)", row=2, col=1)
            fig.add_hline(y=30, line_dash="dash", line_color="green", annotation_text="Oversold (30)", row=2, col=1)
            
            
        return fig

        

# 알림 전략 인터페이스
class NotificationStrategy(ABC):
    @abstractmethod
    def send_notification(self, message: str):
        pass

# 구체적인 알림 전략: 문자 메시지 전송
class SMSNotification(NotificationStrategy):
    def send_notification(self, message: str):
        print(f"문자 메시지 전송: {message}")

        
class PatternDetector(ABC):
    
    @abstractmethod
    def load(self,base_delta, from_date):
        pass
    
    @abstractmethod
    def add_sub_indicator(self,indicator_instance_list):
        pass
    
    @abstractmethod
    def execute(self):
        pass
        
        
# Bitmex 클래스 (컨텍스트)
class Bitmex(PatternDetector):
    #def __init__(self, data_loader: DataLoaderStrategy, processor: DataProcessingStrategy,
    #             visualizer: VisualizationStrategy, notifier: NotificationStrategy):
    #    self.data_loader = data_loader
    #    self.processor = processor
    #    self.visualizer = visualizer
    #    self.notifier = notifier
    #    self.data = None
        
    def __init__(self):
        pass
        
        
    def set_loader(self,data_loader : DataLoaderStrategy):
        self.data_loader = data_loader
    
    def load(self):
        self.data = self.data_loader.load_data()
        #self.data = self.data_loader.pre_precessing(self.data, 15, '2023-12-31 15:01:00')
        
    def set_processor(self, processor: DataProcessingStrategy):
        self.processor = processor
        
    def add_sub_indicator(self,indicator_instance_list):
        '''외부에서 주입받은 DataProcessingStrategy 중, 보조지표 추가하는 작업으로 정의된 클래스를 수행시킨다.'''
        for one_indicator in indicator_instance_list:
            #print("데이터확인")
            #print(self.data)
            self.data = one_indicator.process_data(self.data)
            
    def get_key_points(self, instance : DataProcessingStrategy):
        '''고점을 찾거나 다이버전스를 찾는등의 주요 포인트를 찾을 때 활용'''
        return instance.process_data(self.data) #이미 세팅된 메인데이터를 활용한다.
            
        
    
    def execute(self):
        processed_data = self.processor.process_data(self.data)
        self.visualizer.visualize(processed_data)
        if processed_data[-1] > 3:  # 특정 패턴 감지 예시
            self.notifier.send_notification("패턴이 감지되었습니다!")

# 사용 예시
#if __name__ == "__main__":
#    bitmex = Bitmex(CSVDataLoader(), MovingAverageProcessing(), LineGraphVisualization(), SMSNotification())
#    bitmex.load()


In [34]:
#bitmex = Bitmex(, MovingAverageProcessing(), LineGraphVisualization(), SMSNotification())
#bitmex.load()

In [35]:
bitmex = Bitmex()
#bitmex.set_loader(BitmexCSVDataLoader(15,'2022-12-31 15:01:00'))
bitmex.set_loader(BitmexCSVDataLoader(15,'2023-12-31 15:01:00'))
bitmex.load()

CSV 데이터를 로드하고 Nan을 제거합니다.
입력받은 15분봉으로 2023-12-31 15:01:00 부터 표현합니다.
CSV 데이터 로드 완료.
전처리 완료


In [36]:
bitmex.data

,timestamp_kst,open,low,high,close,timestamp_int
14,2024-01-01 00:15:00,42600.5,42471.5,42607.5,42471.5,1.704036e+09
29,2024-01-01 00:30:00,42471.5,42375.0,42529.5,42516.5,1.704037e+09
44,2024-01-01 00:45:00,42516.5,42430.5,42532.5,42477.5,1.704038e+09
59,2024-01-01 01:00:00,42477.5,42450.0,42576.0,42574.0,1.704038e+09
73,2024-01-01 01:15:00,42572.5,42541.5,42599.5,42577.0,1.704039e+09
...,...,...,...,...,...,...
609200,2025-03-06 17:45:00,91380.0,91075.6,91392.3,91129.0,1.741251e+09
609215,2025-03-06 18:00:00,91129.0,90618.5,91141.3,90779.2,1.741252e+09
609230,2025-03-06 18:15:00,90779.2,90582.0,90991.6,90679.9,1.741252e+09
609245,2025-03-06 18:30:00,90679.9,90557.6,90851.8,90851.8,1.741253e+09


In [37]:
indicator_list = [MovingAverageProcessing(), RSIProcessing()]
bitmex.add_sub_indicator(indicator_list)

이동 평균을 계산했습니다.
RSI를 계산했습니다.


In [38]:
indicator_list_more = [HighPointScoringProcessing([200])]
bitmex.add_sub_indicator(indicator_list_more)

수행한 숫자:40049
가장 높은 점수:200


In [39]:
indicator_list_more = [LowPointScoringProcessing([200])]
bitmex.add_sub_indicator(indicator_list_more)

In [40]:
bitmex.data

,timestamp_kst,open,low,high,close,timestamp_int,ma_20,ma_60,RSI,high_score,low_score
14,2024-01-01 00:15:00,42600.5,42471.5,42607.5,42471.5,1.704036e+09,42471.500,42471.500000,0.000000,0,0
29,2024-01-01 00:30:00,42471.5,42375.0,42529.5,42516.5,1.704037e+09,42494.000,42494.000000,100.000000,0,0
44,2024-01-01 00:45:00,42516.5,42430.5,42532.5,42477.5,1.704038e+09,42488.500,42488.500000,53.571429,0,0
59,2024-01-01 01:00:00,42477.5,42450.0,42576.0,42574.0,1.704038e+09,42509.875,42509.875000,78.393352,0,0
73,2024-01-01 01:15:00,42572.5,42541.5,42599.5,42577.0,1.704039e+09,42523.300,42523.300000,78.746594,0,0
...,...,...,...,...,...,...,...,...,...,...,...
609200,2025-03-06 17:45:00,91380.0,91075.6,91392.3,91129.0,1.741251e+09,91846.505,90883.528333,23.194570,0,0
609215,2025-03-06 18:00:00,91129.0,90618.5,91141.3,90779.2,1.741252e+09,91797.465,90909.848333,15.026236,0,0
609230,2025-03-06 18:15:00,90779.2,90582.0,90991.6,90679.9,1.741252e+09,91753.685,90932.148333,9.268273,0,0
609245,2025-03-06 18:30:00,90679.9,90557.6,90851.8,90851.8,1.741253e+09,91690.895,90962.778333,17.432950,0,0


In [41]:
process_instance = GetHighPoints(bitmex.data, 'high_score', 0.95)
high_points = bitmex.get_key_points(process_instance)
high_points

정형화된 기준값은 : 0.005


d:\python3.9.5\lib\site-packages\pandas\core\indexing.py:1681: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.obj[key] = empty_value
d:\python3.9.5\lib\site-packages\pandas\core\indexing.py:1773: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(ilocs[0], value, pi)
d:\python3.9.5\lib\site-packages\pandas\core\indexing.py:1817: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See t

,timestamp_kst,open,low,high,close,timestamp_int,ma_20,ma_60,RSI,high_score,low_score,normalized_value_high,origin_high_score,high_for_graph
2458,2024-01-02 18:00:00,45809.5,45740.5,45965.5,45762.0,1.704186e+09,45465.450,44668.000000,68.585732,157.0,0,0.785,157,45965.5
2473,2024-01-02 18:15:00,45762.0,45748.0,45916.0,45893.0,1.704187e+09,45488.450,44714.158333,75.875796,1.0,0,0.005,1,45916.0
2488,2024-01-02 18:30:00,45893.0,45682.5,45893.0,45722.0,1.704188e+09,45510.475,44756.058333,65.439300,1.0,0,0.005,1,45893.0
2758,2024-01-02 23:00:00,45733.5,45709.0,45889.5,45845.5,1.704204e+09,45624.275,45360.400000,57.522124,18.0,0,0.090,18,45889.5
2773,2024-01-02 23:15:00,45845.5,45634.5,45883.0,45708.5,1.704205e+09,45615.050,45387.625000,51.969327,1.0,0,0.005,1,45883.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
608945,2025-03-06 13:30:00,91555.5,91532.9,92280.0,92107.6,1.741235e+09,91053.215,90046.221667,74.450426,1.0,0,0.005,1,92280.0
608960,2025-03-06 13:45:00,92107.6,91981.5,92428.7,92200.0,1.741236e+09,91135.105,90092.081667,76.358280,2.0,0,0.010,2,92428.7
608990,2025-03-06 14:15:00,92104.6,92104.6,92482.6,92376.9,1.741238e+09,91321.085,90176.915000,71.151090,1.0,0,0.005,1,92482.6
609005,2025-03-06 14:30:00,92376.9,92310.1,92558.1,92538.8,1.741239e+09,91444.845,90230.728333,66.925363,1.0,0,0.005,1,92558.1


In [42]:
process_instance = GetLowPoints(bitmex.data, 'low_score', 0.95)
low_points = bitmex.get_key_points(process_instance)
low_points

정형화된 기준값은 : 0.005


d:\python3.9.5\lib\site-packages\pandas\core\indexing.py:1681: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.obj[key] = empty_value
d:\python3.9.5\lib\site-packages\pandas\core\indexing.py:1773: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(ilocs[0], value, pi)
d:\python3.9.5\lib\site-packages\pandas\core\indexing.py:1817: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See t

,timestamp_kst,open,low,high,close,timestamp_int,ma_20,ma_60,RSI,high_score,low_score,normalized_value_high,normalized_value_low,origin_low_score,low_for_graph
454,2024-01-01 07:45:00,42514.5,42075.0,42514.5,42282.0,1.704063e+09,42595.475,42582.833333,22.896854,0,30.0,0.0,0.150,30,42075.0
484,2024-01-01 08:15:00,42259.5,42083.5,42262.5,42208.5,1.704064e+09,42558.175,42561.031250,17.844887,0,2.0,0.0,0.010,2,42083.5
499,2024-01-01 08:30:00,42208.5,42185.0,42272.0,42272.0,1.704065e+09,42539.500,42552.272727,27.524893,0,1.0,0.0,0.005,1,42185.0
856,2024-01-01 14:45:00,42324.0,42213.0,42324.0,42266.5,1.704088e+09,42454.975,42510.803571,21.138846,0,23.0,0.0,0.115,23,42213.0
871,2024-01-01 15:00:00,42266.5,42234.5,42330.5,42300.5,1.704089e+09,42446.900,42507.114035,27.449393,0,1.0,0.0,0.005,1,42234.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
605903,2025-03-04 10:45:00,83457.9,82529.5,83745.0,83258.1,1.741053e+09,85822.165,88712.051667,16.803697,0,1.0,0.0,0.005,1,82529.5
605918,2025-03-04 11:00:00,83258.1,82400.1,83767.8,83699.9,1.741054e+09,85690.565,88562.631667,24.425212,0,46.0,0.0,0.230,46,82400.1
606608,2025-03-04 22:30:00,82481.4,82148.3,83440.8,83171.2,1.741095e+09,83663.465,84117.743333,43.532097,0,4.0,0.0,0.020,4,82148.3
606668,2025-03-04 23:30:00,82682.2,81933.6,84967.1,83994.8,1.741099e+09,83498.600,83896.338333,49.953439,0,3.0,0.0,0.015,3,81933.6


In [43]:
draw_instance = BasicPriceWithRsiVisualization()
draw_instance.set_data(bitmex.data, high_points, low_points)
draw_instance.visualize()

NameError: name 'final_low_point' is not defined

In [ ]:
bitmex.data['high_score']